In [ ]:
import os

import h5py

import numpy as np

import astropy.units as u
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

In [ ]:
# Set common directories
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

# Set the names for the different filters in the E24 catalog
filters = ['ACS_F435W', 'ACS_F606W', 'ACS_F775W', 'ACS_F814W', 'ACS_F850LP', 
    'NRC_F090W', 'NRC_F115W', 'NRC_F150W', 'NRC_F200W', 'NRC_F277W', 'NRC_F335M', 'NRC_F356W', 'NRC_F410M', 'NRC_F444W']

# Set the maximum separation between two objects to count as a match
max_sep = 0.1 * u.arcsec

def hist(x, idx, binwidth):

    '''
    Helper function for plotting histograms of photometric property distributions
    '''

    # Create a figure, which doesn't have to be square.
    fig = plt.figure(layout='constrained')

    # Create the main axes
    ax = fig.add_subplot()

    # Create a boolean mask where the AB magnitudes aren't well defined
    mask_finite = np.isfinite(x.value)

    # Create a boolean mask for the indices that have a coordinate match in the JADES dropout selection
    mask_matches = np.isin(np.arange(len(x)), idx)

    # Determine nice limits for the figure
    x_min = min(x[mask_finite]).value
    x_max = max(x[mask_finite]).value
    lim_lower = (int(x_min / binwidth) - 1) * binwidth
    lim_upper = (int(x_max / binwidth) + 1) * binwidth

    # Make histogram bins based on the figure limits and bin size
    bins = np.arange(lim_lower - binwidth, lim_upper + binwidth, binwidth)

    # Set the colors and labels of the two subsamples
    colors = ['red', 'black']
    labels = ['Both', 'Only E24']

    # For the mask of each subsample
    for i, mask in enumerate([mask_matches, ~mask_matches]):

        # Mask out photometry that is non-finite or not part of the subsample
        x_masked = x[mask_finite & mask].value

        # Determine the mean and standard deviation of the subsample
        mean = np.mean(x_masked)
        std = np.std(x_masked)

        # Plot the photometric distribution
        ax.hist(x_masked, bins=bins, color=colors[i], alpha=0.5, label=f'{labels[i]} ($N={len(x_masked)}$, $\mu={mean:.1f}$, $\sigma={std:.1f}$)')

    # Add the y-axis label
    ax.set_ylabel('Count')

    # Add the legend
    ax.legend(loc='upper right')

    # Add an annotation with the number of non-finite photometric measurements
    at = AnchoredText(f'Non-finite: {np.sum(~mask_finite)}', loc='upper left', frameon=False)
    ax.add_artist(at)

    # Create a figure, which doesn't have to be square.
    fig_frac = plt.figure(layout='constrained')

    # Create the main axes
    ax_frac = fig_frac.add_subplot()

    # Calculate the recovery fraction
    fraction = np.histogram(x[mask_finite & mask_matches].value, bins)[0] / np.histogram(x[mask_finite].value, bins)[0]

    # Plot the recovery fraction
    ax_frac.plot((bins[:-1] + bins[1:]) / 2, fraction, ds='steps-mid', color='black')

    # Label the y-axis
    ax_frac.set_ylabel('Fraction recovered')

    # Add an annotation indicating the number of non-finite photometric measurements
    at = AnchoredText(f'Non-finite: {np.sum(~mask_finite)}', loc='upper left', frameon=False)
    ax_frac.add_artist(at)

    return fig, ax, fig_frac, ax_frac

def compare_endsley2024():

    '''
    Make figures comparing the photometric properties of the E24 galaxies that do and do not also appear in the JADES F775W dropout galaxy catalog
    '''

    # Open the E24 catalog of F775W dropouts and get their coordinates
    catalog_endsley2024 = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog_endsley2024)
    coords_endsley2024 = SkyCoord(ra=hdul_endsley2024[1].data['RA'] * u.deg, dec=hdul_endsley2024[1].data['DEC'] * u.deg)

    # Open the JADES catalog of F775W dropouts and get their coordinates
    catalog_jades = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_final.fits'
    hdul_jades = fits.open(catalog_jades)
    coords_jades = SkyCoord(ra=hdul_jades[1].data['RA'] * u.deg, dec=hdul_jades[1].data['DEC'] * u.deg)

    # Get the indices and angular distances of objects in the Endsley et al. (2024) catalog that are the best matches to the galaxies in the JADES F775W dropout selection
    idx_endsley2024, d2d_endsley2024, _ = coords_jades.match_to_catalog_sky(coords_endsley2024)

    # Mask out indices that do not satisfy the maximum separation criterion
    idx_endsley2024 = idx_endsley2024[d2d_endsley2024 < max_sep]

    # For each filter
    for filter in filters:

        # Get the corresponding photometry
        phot_endsley2024_njy = hdul_endsley2024[1].data[filter] * u.nJy
        phot_err_endsley2024_njy = hdul_endsley2024[1].data[f'{filter}_err'] * u.nJy

        # ----------------------------------------------------
        # Plot the photometries against each other
        # ----------------------------------------------------

        # Plot the distribution of the photometry of E24's galaxies that also appear in the JADES dropout selection

        # Convert the photometry to AB magnitudes
        phot_endsley2024_ABmag = phot_endsley2024_njy.to(u.ABmag)

        # Make a histogram of the observed photometry in the filter and the corresponding recovery fraction 
        fig, ax, fig_frac, ax_frac = hist(phot_endsley2024_ABmag, idx_endsley2024, 0.25)

        # Label the axes
        ax.set_xlabel(f'{filter[4:]} (AB mag.) (E24)')
        ax_frac.set_xlabel(f'{filter[4:]} (AB mag.) (E24)')

        # Invert the magnitude axis, so that fainter objects are on the left
        ax.xaxis.set_inverted(True)
        ax_frac.xaxis.set_inverted(True)

        # Save the figures
        fig.savefig(f'{figs}/photometry_comparison/endsley2024/endsley2024_f775w_dropouts_{filter[4:]}.png', dpi=200, bbox_inches='tight')
        fig_frac.savefig(f'{figs}/photometry_comparison/endsley2024/endsley2024_f775w_dropouts_{filter[4:]}_frac.png', dpi=200, bbox_inches='tight')

        plt.close('all')

        # ----------------------------------------------------
        # Plot the SNRs of the photometries against each other
        # ----------------------------------------------------

        # Calculate the filter's SNR
        phot_snr_endsley2024 = phot_endsley2024_njy / phot_err_endsley2024_njy

        # Make histograms of the photometry's SNR and the corresponding recovery fraction
        fig, ax, fig_frac, ax_frac = hist(phot_snr_endsley2024, idx_endsley2024, 1)

        # Label the axes
        ax.set_xlabel(f'SNR({filter[4:]}) (E24)')
        ax_frac.set_xlabel(f'SNR({filter[4:]}) (E24)')
        
        # Save the figures
        fig.savefig(f'{figs}/photometry_comparison/endsley2024/endsley2024_f775w_dropouts_{filter[4:]}_snr.png', dpi=200, bbox_inches='tight')
        fig_frac.savefig(f'{figs}/photometry_comparison/endsley2024/endsley2024_f775w_dropouts_{filter[4:]}_snr_frac.png', dpi=200, bbox_inches='tight')

        plt.close('all')

    # Set the filter pairs to calculate colors for
    colors = [['ACS_F775W','NRC_F090W'], ['NRC_F090W','NRC_F150W'], ['ACS_F606W','NRC_F090W']]

    # For each color
    for color in colors:

        # Get the photometry of the shorter and longer wavelength filter
        phot_sw_endsley2024_njy = hdul_endsley2024[1].data[color[0]] * u.nJy
        phot_lw_endsley2024_njy = hdul_endsley2024[1].data[color[1]] * u.nJy
        
        # Convert the photometry to AB magnitudes
        phot_sw_endsley2024_ABmag = phot_sw_endsley2024_njy.to(u.ABmag)
        phot_lw_endsley2024_ABmag = phot_lw_endsley2024_njy.to(u.ABmag)

        # Calculate the color
        color_endsley2024 = phot_sw_endsley2024_ABmag - phot_lw_endsley2024_ABmag

        # Make histograms of the color distribution and corresponding recovery fraction
        fig, ax, fig_frac, ax_frac = hist(color_endsley2024, idx_endsley2024, 0.25)

        # Label the axes
        ax.set_xlabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (E24)')
        ax_frac.set_xlabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (E24)')

        # Save the figures
        fig.savefig(f'{figs}/photometry_comparison/endsley2024/endsley2024_f775w_dropouts_{color[0][4:]}-{color[1][4:]}.png', dpi=200, bbox_inches='tight')
        fig_frac.savefig(f'{figs}/photometry_comparison/endsley2024/endsley2024_f775w_dropouts_{color[0][4:]}-{color[1][4:]}_frac.png', dpi=200, bbox_inches='tight')

        plt.close('all')

def compare_jades():

    '''
    Make figures comparing the photometric properties of the E24 galaxies that do and do not also appear in the JADES F775W dropout galaxy catalog
    '''

    # Open the E24 catalog of F775W dropouts and get their coordinates
    catalog_endsley2024 = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog_endsley2024)
    coords_endsley2024 = SkyCoord(ra=hdul_endsley2024[1].data['RA'] * u.deg, dec=hdul_endsley2024[1].data['DEC'] * u.deg)

    # Open the JADES catalog of F775W dropouts and get their coordinates
    catalog_jades = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_final.fits'
    hdul_jades = fits.open(catalog_jades)
    coords_jades = SkyCoord(ra=hdul_jades[1].data['RA'] * u.deg, dec=hdul_jades[1].data['DEC'] * u.deg)

    # Get the indices and angular distances of objects in the JADES F775W dropout galaxy catalog that are the best matches to the galaxies in the E24 catalog
    idx_jades, d2d_jades, _ = coords_endsley2024.match_to_catalog_sky(coords_jades)

    # Mask out indices that do not satisfy the maximum separation criterion
    idx_jades = idx_jades[d2d_jades < max_sep]

    # For each filter
    for filter in filters:

        # Get the photometry in the filter from the JADES F775W dropout galaxy catalog
        phot_jades_njy = hdul_jades[1].data[f'{filter[4:]}_KRON_S'] * u.nJy
        phot_err_jades_njy = hdul_jades[1].data[f'{filter[4:]}_KRON_S_e'] * u.nJy

        # ----------------------------------------------------
        # Plot the photometries against each other
        # ----------------------------------------------------

        # Plot the photometric distribution of the photometry of the JADES F775W dropout galaxies that also appear in the E24 catalog

        # Convert the photometry to AB magnitudes
        phot_jades_ABmag = phot_jades_njy.to(u.ABmag)

        # Make histograms of the observed photometry in the filter and the corresponding recovery fraction 
        fig, ax, fig_frac, ax_frac = hist(phot_jades_ABmag, idx_jades, 0.25)

        # Set the axes labels
        ax.set_xlabel(f'{filter[4:]} (AB mag.) (JADES)')
        ax_frac.set_xlabel(f'{filter[4:]} (AB mag.) (JADES)')

        # Invert the magnitude axis so fainter objects are on the left
        ax.xaxis.set_inverted(True)
        ax_frac.xaxis.set_inverted(True)

        # Save the figures
        fig.savefig(f'{figs}/photometry_comparison/jades/jades_f775w_dropouts_{filter[4:]}.png', dpi=200, bbox_inches='tight')
        fig_frac.savefig(f'{figs}/photometry_comparison/jades/jades_f775w_dropouts_{filter[4:]}_frac.png', dpi=200, bbox_inches='tight')

        plt.close('all')

        # ----------------------------------------------------
        # Plot the SNRs of the photometries against each other
        # ----------------------------------------------------

        # Calculate the SNRs in the filter
        phot_snr_jades = phot_jades_njy / phot_err_jades_njy

        # Make histograms comparing the SNRs and recovery fractions
        fig, ax, fig_frac, ax_frac = hist(phot_snr_jades, idx_jades, 1)

        # Set the axes labels
        ax.set_xlabel(f'SNR({filter[4:]}) (JADES)')
        ax_frac.set_xlabel(f'SNR({filter[4:]}) (JADES)')
        
        # Save the figures
        fig.savefig(f'{figs}/photometry_comparison/jades/jades_f775w_dropouts_{filter[4:]}_snr.png', dpi=200, bbox_inches='tight')
        fig_frac.savefig(f'{figs}/photometry_comparison/jades/jades_f775w_dropouts_{filter[4:]}_snr_frac.png', dpi=200, bbox_inches='tight')

        plt.close('all')

    # Set the filter pairs to calculate colors for
    colors = [['ACS_F775W','NRC_F090W'], ['NRC_F090W','NRC_F150W'], ['ACS_F606W','NRC_F090W']]

    # For each color
    for color in colors:

        # Get the shorter and longer wavelength filter's photometry
        phot_sw_jades_njy = hdul_jades[1].data[f'{color[0][4:]}_KRON_S'] * u.nJy
        phot_lw_jades_njy = hdul_jades[1].data[f'{color[1][4:]}_KRON_S'] * u.nJy
        
        # Convert the photometry to AB magnitudes
        phot_sw_jades_ABmag = phot_sw_jades_njy.to(u.ABmag)
        phot_lw_jades_ABmag = phot_lw_jades_njy.to(u.ABmag)

        # Calculate the color
        color_jades = phot_sw_jades_ABmag - phot_lw_jades_ABmag

        # Make histograms comparing the color distributions and recovery fraction
        fig, ax, fig_frac, ax_frac = hist(color_jades, idx_jades, 0.25)

        # Label the axes
        ax.set_xlabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (JADES)')
        ax_frac.set_xlabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (JADES)')

        # Save the figures
        fig.savefig(f'{figs}/photometry_comparison/jades/jades_f775w_dropouts_{color[0][4:]}-{color[1][4:]}.png', dpi=200, bbox_inches='tight')
        fig_frac.savefig(f'{figs}/photometry_comparison/jades/jades_f775w_dropouts_{color[0][4:]}-{color[1][4:]}_frac.png', dpi=200, bbox_inches='tight')

        plt.close('all')

def scatter_hist(x, y, binwidth, fig, ax):

    '''
    A helper function for creating figures with histograms appended to a scatter plot
    '''

    # Set an equal aspect ratio
    ax.set_aspect('equal')

    # Create marginal axes, which have 25% of the size of the main axes.  Note that
    # the inset axes are positioned *outside* (on the right and the top) of the
    # main axes, by specifying axes coordinates greater than 1.  Axes coordinates
    # less than 0 would likewise specify positions on the left and the bottom of
    # the main axes
    ax_histx = ax.inset_axes([0, 1.05, 1, 0.25], sharex=ax)
    ax_histy = ax.inset_axes([1.05, 0, 0.25, 1], sharey=ax)

    # Remove labels from the interior side of the histogram axes
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.tick_params(axis="y", labelleft=False)

    # Add the markers to the scatter plot
    ax.scatter(x, y)

    # Determine nice limits for the figure
    xymin = min(np.min(x), np.min(y))
    xymax = max(np.max(x), np.max(y))
    lim_lower = (int(xymin/binwidth) - 1) * binwidth
    lim_upper = (int(xymax/binwidth) + 1) * binwidth

    # Add the distributions to the histograms
    bins = np.arange(lim_lower - binwidth, lim_upper + binwidth, binwidth)
    ax_histx.hist(x, bins=bins)
    ax_histy.hist(y, bins=bins, orientation='horizontal')

    # Add axis labels to the histograms
    ax_histx.set_ylabel('Count')
    ax_histy.set_xlabel('Count')

    # Calculate the mean and standard deviations of the data
    x_mean = np.mean(x)
    x_std = np.std(x)
    y_mean = np.mean(y)
    y_std = np.std(y)

    # Annotate the statistical information of the data
    at_x = AnchoredText(f'$\mu={x_mean:.1f}$\n$\sigma={x_std:.1f}$', loc='upper right', frameon=False)
    at_y = AnchoredText(f'$\mu={y_mean:.1f}$\n$\sigma={y_std:.1f}$', loc='upper left', frameon=False)
    ax_histx.add_artist(at_x)
    ax_histy.add_artist(at_y)

def compare_endsley2024_vs_jades():

    '''
    Directly compare the photometry of the E24 catalog against the corresponding photometry in the complete JADES catalog
    '''

    # Get the coordinates of objects in the E24 catalog
    catalog_endsley2024 = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog_endsley2024)
    coords_endsley2024 = SkyCoord(ra=hdul_endsley2024[1].data['RA'] * u.deg, dec=hdul_endsley2024[1].data['DEC'] * u.deg)

    # Get the coordinates of objects in the full JADES photometric catalog
    catalog_jades = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits'
    hdul_jades = fits.open(catalog_jades)
    coords_jades = SkyCoord(ra=hdul_jades['KRON_CONV'].data['RA'] * u.deg, dec=hdul_jades['KRON_CONV'].data['DEC'] * u.deg)

    # Find the indices and angular separations of the best matches to E24's galaxies in the JADES catalog
    idx_jades, d2d_jades, _ = coords_endsley2024.match_to_catalog_sky(coords_jades)

    # Get the indices and angular distances of the best-match JADES galaxies in E24's catalog
    idx_endsley2024, d2d_endsley2024, _ = coords_jades[idx_jades].match_to_catalog_sky(coords_endsley2024)

    # Mask out indices that do not satisfy the maximum separation criterion
    idx_jades = idx_jades[d2d_jades < max_sep]
    idx_endsley2024 = idx_endsley2024[d2d_endsley2024 < max_sep]

    print(f'Number of E24 galaxies with a match in JADES: {len(idx_jades)}')

    # For each filter
    for filter in filters:

        # Get the photometry from the E24 catalog
        phot_endsley2024_njy = hdul_endsley2024[1].data[filter][idx_endsley2024] * u.nJy
        phot_err_endsley2024_njy = hdul_endsley2024[1].data[f'{filter}_err'][idx_endsley2024] * u.nJy

        # Get the corresponding photometry from the JADES catalog
        phot_jades_njy = hdul_jades['KRON_CONV'].data[f'{filter[4:]}_KRON_S'][idx_jades] * u.nJy
        phot_err_jades_njy = hdul_jades['KRON_CONV'].data[f'{filter[4:]}_KRON_S_e'][idx_jades] * u.nJy

        # ----------------------------------------------------
        # Plot the photometries against each other
        # ----------------------------------------------------

        # Create a figure, which doesn't have to be square.
        fig = plt.figure(layout='constrained')

        # Create the main axes
        ax = fig.add_subplot()

        # Convert the photometry to AB magnitudes
        phot_endsley2024_ABmag = phot_endsley2024_njy.to(u.ABmag)
        phot_jades_ABmag = phot_jades_njy.to(u.ABmag)

        # Make a mask for finite-valued photometry
        mask_phot = np.isfinite(phot_endsley2024_ABmag.value) & np.isfinite(phot_jades_ABmag.value)

        # Mask out non-finite photometry
        phot_endsley2024_ABmag = phot_endsley2024_ABmag[mask_phot]
        phot_jades_ABmag = phot_jades_ABmag[mask_phot]

        # Draw the scatter plot and histogram distributions
        scatter_hist(phot_jades_ABmag.value, phot_endsley2024_ABmag.value, 0.25, fig, ax)

        # Add a one-to-one line on the plot
        ax.axline((30,30), slope=1, color='red', linestyle='dashed')

        # Label the axes
        ax.set_xlabel(f'{filter[4:]} (AB mag.) (JADES)')
        ax.set_ylabel(f'{filter[4:]} (AB mag.) (E24)')

        # Add an annotation with the number of non-finite points that couldn't be plotted
        at = AnchoredText(f'Non-finite: {np.sum(~mask_phot)}', loc='upper left', frameon=False)
        ax.add_artist(at)

        # Save the figure
        fig.savefig(f'{figs}/photometry_comparison/endsley2024_f775w_dropouts_vs_jades_{filter}.png', bbox_inches='tight', dpi=200)

        plt.close('all')

        # ----------------------------------------------------
        # Plot the SNRs of the photometries against each other
        # ----------------------------------------------------

        # Create a new figure and axes to compare the SNRs
        fig_snr = plt.figure(layout='constrained')
        ax_snr = fig_snr.add_subplot()

        # Calculate the SNRs
        phot_snr_endsley2024 = phot_endsley2024_njy / phot_err_endsley2024_njy
        phot_snr_jades = phot_jades_njy / phot_err_jades_njy

        # Make a mask for non-finite SNRs
        mask_snr = np.isfinite(phot_snr_endsley2024) & np.isfinite(phot_snr_jades)

        # Mask out any non-finite SNRs
        phot_snr_endsley2024 = phot_snr_endsley2024[mask_snr]
        phot_snr_jades = phot_snr_jades[mask_snr]

        # Make a scatter plot with attached histogram distributions of the SNRs
        scatter_hist(phot_snr_jades, phot_snr_endsley2024, 1, fig_snr, ax_snr)

        # Add a one-to-one line to the scatter plot
        ax_snr.axline((0,0), slope=1, color='red', linestyle='dashed')

        # Label the axes
        ax_snr.set_xlabel(f'SNR({filter[4:]}) (JADES)')
        ax_snr.set_ylabel(f'SNR({filter[4:]}) (E24)')

        # Add an annotation indicating the number of objects with non-finite SNRs in at least one of the catalogs
        at = AnchoredText(f'Non-finite: {np.sum(~mask_snr)}', loc='upper left', frameon=False)
        ax_snr.add_artist(at)

        # Save the figure
        fig_snr.savefig(f'{figs}/photometry_comparison/endsley2024_f775w_dropouts_vs_jades_{filter}_snr.png', bbox_inches='tight', dpi=200)

        plt.close('all')

    # Set the filter pairs to calculate colors for
    colors = [['ACS_F775W','NRC_F090W'], ['NRC_F090W','NRC_F150W'], ['ACS_F606W','NRC_F090W']]

    # For each color
    for color in colors:

        # Get the photometry of the filters in the E24 catalog
        phot_sw_endsley2024_njy = hdul_endsley2024[1].data[color[0]][idx_endsley2024] * u.nJy
        phot_lw_endsley2024_njy = hdul_endsley2024[1].data[color[1]][idx_endsley2024] * u.nJy

        # Get the photometry of the filters in the JADES catalog
        phot_sw_jades_njy = hdul_jades['KRON_CONV'].data[f'{color[0][4:]}_KRON_S'][idx_jades] * u.nJy
        phot_lw_jades_njy = hdul_jades['KRON_CONV'].data[f'{color[1][4:]}_KRON_S'][idx_jades] * u.nJy

        # Convert the photometry to AB magnitudes
        phot_sw_endsley2024_ABmag = phot_sw_endsley2024_njy.to(u.ABmag)
        phot_lw_endsley2024_ABmag = phot_lw_endsley2024_njy.to(u.ABmag)
        phot_sw_jades_ABmag = phot_sw_jades_njy.to(u.ABmag)
        phot_lw_jades_ABmag = phot_lw_jades_njy.to(u.ABmag)

        # Calculate the colors
        color_endsley2024 = phot_sw_endsley2024_ABmag - phot_lw_endsley2024_ABmag
        color_jades = phot_sw_jades_ABmag - phot_lw_jades_ABmag

        # Make a mask for non-finite colors
        mask = np.isfinite(color_endsley2024.value) & np.isfinite(color_jades.value)

        # Create a figure, which doesn't have to be square.
        fig = plt.figure(layout='constrained')

        # Create the main axes
        ax = fig.add_subplot()

        # Draw the scatter plot and marginals
        scatter_hist(color_jades.value[mask], color_endsley2024.value[mask], 0.25, fig, ax)#, ax, ax_histx, ax_histy)

        # Add a one-to-one line to the scatter plot
        ax.axline((1,1), slope=1, color='red', linestyle='dashed')

        # Add labels to the axes
        ax.set_xlabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (JADES)')
        ax.set_ylabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (E24)')

        # Add an annotation indicating the number of objects with a non-finite color in one of the catalogs
        at = AnchoredText(f'Non-finite: {np.sum(~mask)}', loc='upper left', frameon=False)
        ax.add_artist(at)

        # Save the figure
        fig.savefig(f'{figs}/photometry_comparison/endsley2024_f775w_dropouts_vs_jades_{color[0]}_{color[1]}_color.png', bbox_inches='tight', dpi=200)

        plt.close('all')

def compare_fails():

    '''
    Compare the JADES and Endsley et al. (2024) photometry of galaxies from the Endsley et al. (2024) F775W dropout sample that do not also appear in my JADES selection
    '''

    # Get the coordinates of the F775W dropouts from Endsley et al. (2024)
    catalog_endsley2024 = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog_endsley2024)
    coords_endsley2024 = SkyCoord(ra=hdul_endsley2024[1].data['RA'] * u.deg, dec=hdul_endsley2024[1].data['DEC'] * u.deg)

    # Get the coordinates of the galaxies of the merged JADES GOODS-N and GOODS-S catalog
    catalog_jades = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits'
    hdul_jades = fits.open(catalog_jades)
    coords_jades = SkyCoord(ra=hdul_jades['KRON_CONV'].data['RA'] * u.deg, dec=hdul_jades['KRON_CONV'].data['DEC'] * u.deg)

    # Get the coordinates of the galaxies in my JADES selection of F775W dropouts
    catalog_jades_dropouts = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_final.fits'
    hdul_jades_dropouts = fits.open(catalog_jades_dropouts)
    coords_jades_dropouts = SkyCoord(ra=hdul_jades_dropouts[1].data['RA'] * u.deg, dec=hdul_jades_dropouts[1].data['DEC'] * u.deg)

    # Find the best matches in the E24 catalog to the galaxies in my JADES selection of F775W dropouts
    idx_endsley2024, d2d_endsley2024, _ = coords_jades_dropouts.match_to_catalog_sky(coords_endsley2024)

    # Mask out indices that are not coordinate matches
    idx_endsley2024 = idx_endsley2024[d2d_endsley2024 < max_sep]

    # Get the coordinates of the galaxies from E24 that do not also appear in my JADES selection of F775W dropouts
    coords_endsley2024_fails = np.delete(coords_endsley2024, idx_endsley2024)

    # Get the indices and angular separation of the best matches in the JADES catalog to the E24 galaxies not in the JADES F775W dropout galaxy catalog
    idx_jades_fails, d2d_jades_fails, _ = coords_endsley2024_fails.match_to_catalog_sky(coords_jades)

    # Make a mask of the galaxies with a match
    mask_matches = d2d_jades_fails < max_sep

    # Mask the indices by that mask, so only galaxies in the JADES catalog that also appear in the E24 catalog but not the JADES F775W dropout catalog remain
    idx_jades_fails = idx_jades_fails[mask_matches]

    # Make a dictionary of all the E24 photometry of the galaxies that do not also appear in my JADES selection of F775W dropouts
    phot_endsley2024_fails_full = {filter: np.delete(hdul_endsley2024[1].data[filter], idx_endsley2024) for filter in filters + [f'{filter}_err' for filter in filters]}

    # Set the filter pairs to calculate colors for
    colors = [['ACS_F775W','NRC_F090W'], ['NRC_F090W','NRC_F150W'], ['ACS_F606W','NRC_F090W']]

    # Is the below loop redundant? Appears to be doing the same thing for the filters used in the colors, as was already set above?

    # For each color
    for color in colors:

        phot_endsley2024_fails_full[color[0]] = np.delete(hdul_endsley2024[1].data[color[0]], idx_endsley2024)
        phot_endsley2024_fails_full[color[1]] = np.delete(hdul_endsley2024[1].data[color[1]], idx_endsley2024)

    # Make a pared-down dictionary with just the entries that also have a match in the JADES catalog
    phot_endsley2024_fails_filtered = {key: val[mask_matches] for key, val in phot_endsley2024_fails_full.items()}

    # For each filter
    for filter in filters:

        # Get the photometry from the E24 catalog
        phot_endsley2024_njy = phot_endsley2024_fails_filtered[filter] * u.nJy
        phot_err_endsley2024_njy = phot_endsley2024_fails_filtered[f'{filter}_err'] * u.nJy

        # Get the photometry from the JADES catalog
        phot_jades_njy = hdul_jades['KRON_CONV'].data[f'{filter[4:]}_KRON_S'][idx_jades_fails] * u.nJy
        phot_err_jades_njy = hdul_jades['KRON_CONV'].data[f'{filter[4:]}_KRON_S_e'][idx_jades_fails] * u.nJy

        # ----------------------------------------------------
        # Plot the photometries against each other
        # ----------------------------------------------------

        # Create a figure, which doesn't have to be square.
        fig = plt.figure(layout='constrained')

        # Create the main axes
        ax = fig.add_subplot()

        # Convert the photometry to AB magnitudes
        phot_endsley2024_ABmag = phot_endsley2024_njy.to(u.ABmag)
        phot_jades_ABmag = phot_jades_njy.to(u.ABmag)

        # Create a mask for the non-finite photometry
        mask_phot = np.isfinite(phot_endsley2024_ABmag.value) & np.isfinite(phot_jades_ABmag.value)

        # Mask out any non-finite photometry
        phot_endsley2024_ABmag = phot_endsley2024_ABmag[mask_phot]
        phot_jades_ABmag = phot_jades_ABmag[mask_phot]

        # Draw the scatter plot and marginals
        scatter_hist(phot_jades_ABmag.value, phot_endsley2024_ABmag.value, 0.25, fig, ax)

        # Add a one-to-one line to the scatter plot
        ax.axline((30,30), slope=1, color='red', linestyle='dashed')

        # Label the axes
        ax.set_xlabel(f'{filter[4:]} (AB mag.) (JADES)')
        ax.set_ylabel(f'{filter[4:]} (AB mag.) (E24)')

        # Add a label annotating the number of objects which had at least one non-finite entry in one of the catalogs
        at = AnchoredText(f'Non-finite: {np.sum(~mask_phot)}', loc='upper left', frameon=False)
        ax.add_artist(at)

        # Save the figure
        fig.savefig(f'{figs}/photometry_comparison/endsley2024_f775w_dropouts_vs_jades_fails_{filter}.pdf', bbox_inches='tight')

        plt.close('all')

        # ----------------------------------------------------
        # Plot the SNRs of the photometries against each other
        # ----------------------------------------------------

        # Make a new figure and axes for the filter's SNR
        fig_snr = plt.figure(layout='constrained')
        ax_snr = fig_snr.add_subplot()

        # Calculate the SNRs in the filter
        phot_snr_endsley2024 = phot_endsley2024_njy / phot_err_endsley2024_njy
        phot_snr_jades = phot_jades_njy / phot_err_jades_njy

        # Create a mask for finite SNRs
        mask_snr = np.isfinite(phot_snr_endsley2024) & np.isfinite(phot_snr_jades)

        # Mask out any non-finite SNRs
        phot_snr_endsley2024 = phot_snr_endsley2024[mask_snr]
        phot_snr_jades = phot_snr_jades[mask_snr]

        # Make a figure with a scatter plot and histograms of the photometry
        scatter_hist(phot_snr_jades, phot_snr_endsley2024, 1, fig_snr, ax_snr)

        # Make a one-to-one line on the scatter plot
        ax_snr.axline((0,0), slope=1, color='red', linestyle='dashed')

        # Label the axes
        ax_snr.set_xlabel(f'SNR({filter[4:]}) (JADES)')
        ax_snr.set_ylabel(f'SNR({filter[4:]}) (E24)')

        # Add a label indicating the number of objects with a non-finite SNR in at least one of the catalogs
        at = AnchoredText(f'Non-finite: {np.sum(~mask_snr)}', loc='upper left', frameon=False)
        ax_snr.add_artist(at)

        # Save the figure
        fig_snr.savefig(f'{figs}/photometry_comparison/endsley2024_f775w_dropouts_vs_jades_fails_{filter}_snr.pdf', bbox_inches='tight')

        plt.close('all')

    # Set the filter pairs to calculate colors for
    colors = [['ACS_F775W','NRC_F090W'], ['NRC_F090W','NRC_F150W'], ['ACS_F606W','NRC_F090W']]

    # For each color
    for color in colors:

        # Get the photometry of the short and long wavelength filters from the E24 catalog
        phot_sw_endsley2024_njy = phot_endsley2024_fails_filtered[color[0]] * u.nJy
        phot_lw_endsley2024_njy = phot_endsley2024_fails_filtered[color[1]] * u.nJy

        # Get the photometry of the short and long wavelength filters from the JADES catalog
        phot_sw_jades_njy = hdul_jades['KRON_CONV'].data[f'{color[0][4:]}_KRON_S'][idx_jades_fails] * u.nJy
        phot_lw_jades_njy = hdul_jades['KRON_CONV'].data[f'{color[1][4:]}_KRON_S'][idx_jades_fails] * u.nJy

        # Convert the photometry to AB magnitudes
        phot_sw_endsley2024_ABmag = phot_sw_endsley2024_njy.to(u.ABmag)
        phot_lw_endsley2024_ABmag = phot_lw_endsley2024_njy.to(u.ABmag)
        phot_sw_jades_ABmag = phot_sw_jades_njy.to(u.ABmag)
        phot_lw_jades_ABmag = phot_lw_jades_njy.to(u.ABmag)

        # Calculate the colors
        color_endsley2024 = phot_sw_endsley2024_ABmag - phot_lw_endsley2024_ABmag
        color_jades = phot_sw_jades_ABmag - phot_lw_jades_ABmag

        # Create a mask for any non-finite colors
        mask = np.isfinite(color_endsley2024.value) & np.isfinite(color_jades.value)

        # Create a figure, which doesn't have to be square.
        fig = plt.figure(layout='constrained')

        # Create the main axes
        ax = fig.add_subplot()

        # Draw the scatter plot and marginals
        scatter_hist(color_jades.value[mask], color_endsley2024.value[mask], 0.25, fig, ax)

        # Add a one-to-one line
        ax.axline((1,1), slope=1, color='red', linestyle='dashed')

        # Add labels to the axes
        ax.set_xlabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (JADES)')
        ax.set_ylabel(f'{color[0][4:]}$-${color[1][4:]} (AB mag.) (E24)')

        # Add a label indicating the number of objects with a non-finite color in either catalog
        at = AnchoredText(f'Non-finite: {np.sum(~mask)}', loc='upper left', frameon=False)
        ax.add_artist(at)

        # Save the figure
        fig.savefig(f'{figs}/photometry_comparison/endsley2024_f775w_dropouts_vs_jades_fails_{color[0]}_{color[1]}_color.pdf', bbox_inches='tight')

        plt.close('all')

def compare_conds_fails():

    '''
    Investigate the JADES photometry of galaxies from the Endsley et al. (2024) F775W dropout sample that do not also appear in my JADES selection.
    This is slightly different from compare_fails() because this analyzes the photometry as it was selected (since that involved some minor alterations), not as cataloged 
    from the original JADES or Endsley et al. (2024) catalogs
    '''

    # Get the coordinates of the F775W dropouts from E24
    catalog_endsley2024 = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog_endsley2024)
    coords_endsley2024 = SkyCoord(ra=hdul_endsley2024[1].data['RA'] * u.deg, dec=hdul_endsley2024[1].data['DEC'] * u.deg)

    # Get the coordinates of the galaxies of the merged JADES GOODS-N and GOODS-S catalog
    catalog_jades = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits'
    hdul_jades = fits.open(catalog_jades)
    coords_jades = SkyCoord(ra=hdul_jades['KRON_CONV'].data['RA'] * u.deg, dec=hdul_jades['KRON_CONV'].data['DEC'] * u.deg)

    # Get the coordinates of the galaxies in my JADES selection of F775W dropouts
    catalog_jades_dropouts = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_final.fits'
    hdul_jades_dropouts = fits.open(catalog_jades_dropouts)
    coords_jades_dropouts = SkyCoord(ra=hdul_jades_dropouts[1].data['RA'] * u.deg, dec=hdul_jades_dropouts[1].data['DEC'] * u.deg)

    # Find the best matches in the E24 catalog to the galaxies in my JADES F775W dropout galaxy selection
    idx_both, d2d_both, _ = coords_jades_dropouts.match_to_catalog_sky(coords_endsley2024)

    # Mask out indices that are not coordinate matches
    idx_both = idx_both[d2d_both < max_sep]

    # Get the coordinates of the galaxies that do not also appear in my JADES F775W dropout galaxy selection
    coords_endsley2024_fails = np.delete(coords_endsley2024, idx_both)

    # Get the indices and angular separation of the best matches in the JADES catalog to the E24 galaxies not in the JADES F775W dropout galaxy catalog
    idx_jades_fails, d2d_jades_fails, _ = coords_endsley2024_fails.match_to_catalog_sky(coords_jades)

    # Mask out indices that do not have a match
    idx_jades_fails = idx_jades_fails[d2d_jades_fails < max_sep]

    # Open the merged catalog's photometry as a table
    t = Table(hdul_jades['KRON_CONV'].data)

    # Set the F775W flux density to the 1 standard deviation upper limit if the SNR is < 1
    t['F775W_KRON_S'] = np.where(t['F775W_KRON_S'] / t['F775W_KRON_S_e'] < 1, t['F775W_KRON_S_e'], t['F775W_KRON_S'])

    # Before adjusting the F606W photometry, calculate the F606W SNR. This is necessary to do before the next operation assigning upper limits 
    # to low SNR F606W photometry, which would alter the F606W SNR if measured after that operation.
    f606w_snr = t['F606W_KRON_S'] / t['F606W_KRON_S_e']

    # Set the F606W flux density to the 1 standard deviation upper limit if the SNR is < 1
    t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] / t['F606W_KRON_S_e'] < 1, t['F606W_KRON_S_e'], t['F606W_KRON_S'])

    # Calculate AB magnitudes for photometry used to measure colors. Use the unitless values since including units has been troublesome.
    f606w_ABmag = (t['F606W_KRON_S'] * u.nJy).to(u.ABmag).value
    f775w_ABmag = (t['F775W_KRON_S'] * u.nJy).to(u.ABmag).value
    f090w_ABmag = (t['F090W_KRON_S'] * u.nJy).to(u.ABmag).value
    f150w_ABmag = (t['F150W_KRON_S'] * u.nJy).to(u.ABmag).value

    # ---------------------------------------------------------------------------
    # Plot the color distributions of the three main photometric color selections
    # ---------------------------------------------------------------------------

    # Calculate the colors used for the photometric color selections
    f775w_break_color = (f775w_ABmag - f090w_ABmag)[idx_jades_fails]
    uv_cont_color = (f090w_ABmag - f150w_ABmag)[idx_jades_fails]
    color_3 = (f775w_ABmag - 2 * f090w_ABmag + f150w_ABmag)[idx_jades_fails]

    # Assemble the colors into a list and set the labels and critical thresholds of the colors
    colors = [f775w_break_color, uv_cont_color, color_3]
    labels = ['F775W - F090W', 'F090W - F150W', 'F775W - 2 F090W + F150W']
    thresholds = [1.2, 1.0, 1.2]

    # For each color
    for i, color in enumerate(colors):

        # Remove non-finite values
        color = color[np.isfinite(color)]

        # Make a new figure
        fig, ax = plt.subplots()

        # Make a histogram of the color distribution
        ax.hist(color, bins=np.arange(np.min(color) - 0.25, np.max(color) + 0.25, 0.25))

        # Label the axes
        ax.set_xlabel(f'{labels[i]} (AB mag.) (JADES)')
        ax.set_ylabel('Count')

        # Make a vertical line indicating the cutoff of the condition
        ax.axvline(thresholds[i], c='black', ls='dashed')

        # Annotate the figure with the number of non-finite colors and the number that fail the condition
        at = AnchoredText(f'Non-finite: {int(len(idx_jades_fails) - len(color))}', #\nFails: {np.sum(color < 1.2)}', 
            loc='upper right', frameon=False)
        ax.add_artist(at)

        # Save the figure
        fig.savefig(f'{figs}/photometry_comparison/fails/{labels[i]}.png', bbox_inches='tight', dpi=200)

        plt.close('all')

    # ------------------------------------------------------
    # Plot the color distribution of the F606W - F090W break
    # ------------------------------------------------------

    # Calculate the F606W break color
    f606w_break_color = (f606w_ABmag - f090w_ABmag)[idx_jades_fails]
    color = f606w_break_color[np.isfinite(f606w_break_color)]

    # Make a new figure
    fig, ax = plt.subplots()

    # Make a histogram of the color for objects with F775W - F090W > 2.5 (in this case, the exact F606W - F090W color is disregarded)
    ax.hist(color[f775w_break_color[np.isfinite(f606w_break_color)] > 2.5], bins=np.arange(np.min(color) - 0.25, np.max(color) + 0.25, 0.25), 
        label='F775W - F090W > 2.5', color='red', alpha=0.5)

    # Make histograms of the color for objects not disregarded, based on if the SNR(F606W) < 2, which changes the break condition
    ax.hist(color[(f775w_break_color[np.isfinite(f606w_break_color)] <= 2.5) & (f606w_snr[idx_jades_fails][np.isfinite(f606w_break_color)] >= 2)], bins=np.arange(np.min(color) - 0.25, np.max(color) + 0.25, 0.25), 
        label=r'SNR(F606W)$\geq$2', color='blue', alpha=0.5)
    ax.hist(color[(f775w_break_color[np.isfinite(f606w_break_color)] <= 2.5) & (f606w_snr[idx_jades_fails][np.isfinite(f606w_break_color)] < 2)], bins=np.arange(np.min(color) - 0.25, np.max(color) + 0.25, 0.25), 
        label='SNR(F606W) < 2', color='green', alpha=0.5)

    # Label the axes
    ax.set_xlabel(f'F606W - F090W (AB mag.) (JADES)')
    ax.set_ylabel('Count')

    # Make a vertical line indicating the cutoff of the conditions
    ax.axvline(2.7, c='blue', ls='dashed')
    ax.axvline(1.8, c='green', ls='dashed')

    # Annotate the figure with the number of non-finite colors and the number that fail the condition
    at = AnchoredText(f'Non-finite: {int(len(idx_jades_fails) - len(color))}',
        loc='upper right', frameon=False)
    ax.add_artist(at)

    # Add a legend
    ax.legend(loc='upper left')

    # Save the figure
    fig.savefig(f'{figs}/photometry_comparison/fails/f606w_break.png', bbox_inches='tight', dpi=200)

    plt.close('all')

    # -------------------------------
    # Plot the F435W SNR distribution
    # -------------------------------

    # Calculate the F435W SNR
    f435w_snr = (t['F435W_KRON_S'] / t['F435W_KRON_S_e'])[idx_jades_fails]
    snr = f435w_snr[np.isfinite(f435w_snr)]

    # Make a new figure
    fig, ax = plt.subplots()

    # Make a histogram of the SNR for objects with F775W - F090W > 2.5 (in this case, the exact SNR is disregarded)
    ax.hist(snr[f775w_break_color[np.isfinite(f435w_snr)] > 2.5], bins=np.arange(np.min(snr) - 1, np.max(snr) + 1, 1), 
        label='F775W - F090W > 2.5', color='red', alpha=0.5)

    # Make a histogram of the SNR for objects with with F775W - F090W <= 2.5
    ax.hist(snr[f775w_break_color[np.isfinite(f435w_snr)] <= 2.5], bins=np.arange(np.min(snr) - 1, np.max(snr) + 1, 1), 
        label=r'F775W - F090W $\leq$ 2.5', color='blue', alpha=0.5)

    # Label the axes
    ax.set_xlabel(f'SNR(F435W) (JADES)')
    ax.set_ylabel('Count')

    # Make a vertical line indicating the cutoff of the conditions
    ax.axvline(2, c='black', ls='dashed')

    # Annotate the figure with the number of non-finite colors and the number that fail the condition
    at = AnchoredText(f'Non-finite: {int(len(idx_jades_fails) - len(snr))}',
        loc='upper right', frameon=False)
    ax.add_artist(at)

    # Add a legend
    ax.legend(loc='upper left')

    # Save the figure
    fig.savefig(f'{figs}/photometry_comparison/fails/f435w_snr_gtr_2.png', bbox_inches='tight', dpi=200)

    plt.close('all')

    # -------------------------------------------
    # Plot the F814W and F850LP SNR distributions
    # -------------------------------------------

    # Set the strings of the filters
    filters = ['F814W', 'F850LP']

    # Make a new figure
    fig, ax = plt.subplots()

    # For each filter
    for filter in filters:

        # Calculate the SNR
        snr = (t[f'{filter}_KRON_S'] / t[f'{filter}_KRON_S_e'])[idx_jades_fails]
        snr = snr[np.isfinite(snr)]

        # Make a histogram of the SNR for objects with F775W - F090W > 2.5 (in this case, the exact SNR is disregarded)
        ax.hist(snr, bins=np.arange(np.min(snr) - 1, np.max(snr) + 1, 1), 
            label=filter, alpha=0.5)

    # Label the axes
    ax.set_xlabel(f'SNR (JADES)')
    ax.set_ylabel('Count')

    # Make a vertical line indicating the cutoff of the conditions
    ax.axvline(3, c='black', ls='dashed')

    # Annotate the figure with the number of non-finite colors and the number that fail the condition
    at = AnchoredText(f'Non-finite: {int(len(idx_jades_fails) - len(snr))}',
        loc='upper right', frameon=False)
    ax.add_artist(at)

    # Add a legend
    ax.legend(loc='upper left')

    # Save the figure
    fig.savefig(f'{figs}/photometry_comparison/fails/f814w_f850lp_snr_gtr_3.png', bbox_inches='tight', dpi=200)

    plt.close('all')

    # -----------------------------------------------------------
    # Continue with constructing the various selection conditions
    # -----------------------------------------------------------

    # Make a boolean mask for a break in F775W
    cond_f775w_break = (f775w_ABmag - f090w_ABmag) > 1.2

    # Make a boolean mask for a much flatter near-IR 
    cond_flat = (f090w_ABmag - f150w_ABmag) < 1.0

    # Make a boolean mask for a much sharper F775W break than the observed near-IR slope
    cond_f775w_break_gtr = (f775w_ABmag - f090w_ABmag) > (f090w_ABmag - f150w_ABmag + 1.2)

    # Make a boolean mask for a low SNR in F435W. At z ~ 6, the Lyman break is completely longward of F435W, so this filter should have minimal 
    # flux for true Lyman break galaxies.
    cond_lyc_snr = (t['F435W_KRON_S'] / t['F435W_KRON_S_e']) < 2

    # Make a boolean mask for a strong break in F606W, which z ~ 6 galaxies should show. If the SNR in F606W is low (< 2), relax the necessary 
    # break condition.
    cond_f606w_break = (f606w_ABmag - f090w_ABmag) > 2.7
    cond_f606w_break_weak = (f606w_ABmag - f090w_ABmag) > 1.8
    cond_f606w_break = np.where(f606w_snr < 2, cond_f606w_break_weak, cond_f606w_break)

    # Make a boolean mask for strong F775W breaks
    cond_f775w_break_strong = (f775w_ABmag - f090w_ABmag) > 2.5

    # ------------------------------------------------------
    # Create boolean masks of the photometric SNR selections
    # ------------------------------------------------------

    # Make a list of JWST filters in the joint JADES GOODS-N / GOODS-S catalog
    jwst_filters = ['F090W','F115W','F150W','F200W','F277W','F335M','F356W','F410M','F444W']

    # Make empty lists of the JWST photometry and uncertainties corresponding to the filters, to be filled in the loop below
    jwst_photometry = []
    jwst_photometry_e = []

    # For each JWST filter in the catalog
    for filter in jwst_filters:

        # Add the filter's photometry to the lists
        jwst_photometry.append([t[f'{filter}_KRON_S'].data])
        jwst_photometry_e.append([t[f'{filter}_KRON_S_e'].data])

    # Stack the JWST photometry and uncertainties
    jwst_photometry = np.vstack(jwst_photometry)
    jwst_photometry_e = np.vstack(jwst_photometry_e)

    # Calculate the SNR of the JWST photometry
    jwst_photometry_snr = jwst_photometry / jwst_photometry_e

    # Make a boolean mask for objects with any JWST filter with SNR > 5
    cond_snr_gtr_5 = np.any(jwst_photometry_snr > 5, axis=0)

    # Make a boolean mask for objects with at least 3 JWST filters with SNR > 3
    cond_3_snr_gtr_3 = np.sum(jwst_photometry_snr > 3, axis=0) >= 3

    # Make boolean masks for objects with F814W / F850LP SNRs > 3
    cond_f814w_snr_gtr_3 = (t['F814W_KRON_S'] / t['F814W_KRON_S_e']) > 3
    cond_f850lp_snr_gtr_3 = (t['F850LP_KRON_S'] / t['F850LP_KRON_S_e']) > 3

    # ---------------------------------------------------
    # Create boolean masks for artifacts or contamination
    # ---------------------------------------------------

    # Make a boolean mask of objects with finite uncertainties, where appropriate
    cond_real_errors = ~np.isnan(t['F775W_KRON_S_e']) & (~np.isnan(t['F814W_KRON_S_e']) | ~np.isnan(t['F850LP_KRON_S_e'])) & ~np.isnan(t['F090W_KRON_S_e']) & ~np.isnan(t['F150W_KRON_S_e'])

    # Make a boolean mask of non-stars
    cond_star = Table(hdul_jades['FLAG'].data)['FLAG_ST'] != 1

    # Make a boolean mask of objects not contaminated by bright stars
    cond_bs = Table(hdul_jades['FLAG'].data)['FLAG_BS'] != 1.0

    # Make a boolean mask of objects that don't have a bright neighbor >10x as bright
    cond_bn = Table(hdul_jades['FLAG'].data)['FLAG_BN'] != 2.0

    # --------------------------------------------------------------------
    # Create a boolean mask for the sensitivity of the rest-optical colors
    # --------------------------------------------------------------------

    # Open the file storing the rest-UV flux densities (measured in the observed frame)
    with h5py.File(f'{results}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init_f1500.h5', 'r') as file:

        # Get the IDs of the JADES galaxies in E24 but not recovered by this work's JADES F775W dropout galaxy sample
        ids_jades_fails = t['ID'][idx_jades_fails]

        # Make an empty list to append the measured flux densities to
        f_1500s = []

        # For each ID
        for id in ids_jades_fails:

            # If the object was fitted by Bagpipes (which may not always be the case, if it failed one of the initial conditions first)
            if str(id) in file:

                # Get the group in the HDF5 file from the ID
                group = file[str(id)]

                # Calculate the median flux density
                f_1500 = np.nanmedian(group['flux densities'][:])# * u.nJy
            
            else:

                # Assign a dummy value to the flux density which should also fail
                f_1500 = np.nan# * u.nJy
            
            # Append the flux density
            f_1500s.append(f_1500)

    # Assemble the boolean condition to check if the rest-optical photometry is sufficiently sensitive
    cond_opt_sen = ((f_1500s / (t['F335M_KRON_e'][idx_jades_fails])) > 3) \
        & ((f_1500s / (t['F356W_KRON_e'][idx_jades_fails])) > 3) \
        & ((f_1500s / (t['F410M_KRON_e'][idx_jades_fails])) > 3) \
        & ((f_1500s / (t['F444W_KRON_e'][idx_jades_fails])) > 3)

    # The above boolean condition is only as long as the number of E24 galaxies present in the JADES catalog that were not recovered by this work's F775W dropout selection. This means
    # that to properly apply the boolean with the other conditions, it must be adjusted and the values placed in the correct places

    # Make a new, empty boolean with the same length as the JADES catalog
    cond_opt = np.zeros(len(t), dtype=bool)

    # Set the indices of the E24 galaxies present in the JADES catalog but not recovered by this work's F775W dropout selection to the boolean value calculated previously
    cond_opt[idx_jades_fails] = cond_opt_sen

    # ----------------------------------------------
    # Combine and apply the individual boolean masks
    # ----------------------------------------------

    # Combine all the boolean masks into a single mask
    conditions = [cond_opt, cond_star, cond_bs, cond_bn, cond_real_errors, cond_f775w_break, cond_flat, cond_f775w_break_gtr, ((cond_lyc_snr & cond_f606w_break) | cond_f775w_break_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]

    # -----------------------------------------------------------------------------
    # Calculate and plot the cumulative number of conditions that each object fails
    # -----------------------------------------------------------------------------

    # Make an empty (all false) boolean array for the table rows
    bool_jades = np.zeros(len(t), dtype=bool)

    # Set the indices of the boolean array for objects that appear in the E24 catalog, and have a match in JADES, but do not appear in the JADES F775W dropout galaxy catalog, to true
    bool_jades[idx_jades_fails] = True

    # Make a zero-filled array to contain the number of conditions that each E24 object not recovered in the JADES dropout sample but present in the JADES catalog fails
    fails = np.zeros((len(conditions), len(t[bool_jades])), dtype=int)

    # Set labels explaining the various conditions
    labels = ['Insufficiently sensitive rest-optical photometry', 'Stars', 'Bright star contamination', 'Bright neighbor (> 10x)', 'Non-finite uncertainties', 'Insufficient F775W break', 
        'F090W - F150W too blue', 'Insufficient F775W break compared to F090W - F150W', 'F435W SNR too high and insufficient F606W break, or insufficient F775W break',
        'No SNR > 5 NIRCam photometry', 'Insufficient (<3) SNR > 3 NIRCam photometry', 'No SNR > 3 photometry in F814W or F850LP']

    # For each boolean condition
    for i, condition in enumerate(conditions):

        # Store the outcome of the condition for each object
        fails[i,:] = (~condition).astype(int)[bool_jades]

    # For each object's fails
    for i, obj in enumerate(fails.T):

        # If the object failed one of the initial conditions tested
        if np.sum(obj[1:]) > 0:

            # Set the final conditions tested as a 'success', since it was not even tested because it failed the first set of conditions. When counting
            # failures, I think calling the second set a failure because it was not tested is not appropriate
            fails[0,i] = 0

    # For each boolean condition
    for i, condition in enumerate(conditions):

        # Print the condition description and number of objects that failed that condition
        print(f'{labels[i]}: \t{np.sum(fails, axis=1)[i]}')

    # Make a new figure to plot the distribution of cumulative condition failures
    fig, ax = plt.subplots()

    # Add a histogram plotting the distribution of cumulative condition failures for each object
    ax.hist(np.sum(fails, axis=0), bins=np.arange(-0.5, len(conditions) + 1, 1))

    # Label the axes
    ax.set_xlabel('Failed conditions')
    ax.set_ylabel('Count')

    # Add a title
    ax.set_title('E24 galaxies absent from this work\'s JADES\nF775W dropout galaxy selection')

    # Save the figure
    fig.savefig(f'{figs}/photometry_comparison/fails/endsley2024_total_fails.png', bbox_inches='tight', dpi=200)

    plt.close('all')

In [ ]:
compare_endsley2024()

In [ ]:
compare_jades()

In [ ]:
compare_endsley2024_vs_jades()

Number of E24 galaxies with a match in JADES: 268


/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/function/logarithmic.py:67: RuntimeWarning: invalid value encountered in log10
  return dex.to(self._function_unit, np.log10(x))
/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/function/logarithmic.py:67: RuntimeWarning: divide by zero encountered in log10
  return dex.to(self._function_unit, np.log10(x))
/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/quantity.py:659: RuntimeWarning: invalid value encountered in subtract
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


In [ ]:
compare_fails()

In [ ]:
compare_conds_fails()

/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/function/logarithmic.py:67: RuntimeWarning: invalid value encountered in log10
  return dex.to(self._function_unit, np.log10(x))
/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/function/logarithmic.py:67: RuntimeWarning: divide by zero encountered in log10
  return dex.to(self._function_unit, np.log10(x))
/var/folders/md/b3rxd7_x6zgd55s__25hxcwh0000gn/T/ipykernel_34970/255422601.py:767: RuntimeWarning: invalid value encountered in subtract
  uv_cont_color = (f090w_ABmag - f150w_ABmag)[idx_jades_fails]
/var/folders/md/b3rxd7_x6zgd55s__25hxcwh0000gn/T/ipykernel_34970/255422601.py:768: RuntimeWarning: invalid value encountered in add
  color_3 = (f775w_ABmag - 2 * f090w_ABmag + f150w_ABmag)[idx_jades_fails]
/var/folders/md/b3rxd7_x6zgd55s__25hxcwh0000gn/T/ipykernel_34970/255422601.py:930: RuntimeWarning: invalid value encountered in subtract
  cond_flat = (f090w_ABmag - f150

Insufficiently sensitive rest-optical photometry: 	39
Stars: 	1
Bright star contamination: 	1
Bright neighbor (> 10x): 	17
Non-finite uncertainties: 	5
Insufficient F775W break: 	20
F090W - F150W too blue: 	9
Insufficient F775W break compared to F090W - F150W: 	31
F435W SNR too high and insufficient F606W break, or insufficient F775W break: 	31
No SNR > 5 NIRCam photometry: 	4
Insufficient (<3) SNR > 3 NIRCam photometry: 	4
No SNR > 3 photometry in F814W or F850LP: 	55
